https://karan3-zoh.medium.com/paper-summary-imagenet-classification-with-deep-convolutional-neural-networks-41ce6c65960

In [1]:
import torch.nn as nn
import torch.nn.functional as F
from torchinfo import summary
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2
from torch.optim import Adam
from sklearn.metrics import accuracy_score

In [2]:
device = torch.device('cuda') if torch.cuda.is_available else torch.device('cpu')

In [15]:
class AlexNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=96, kernel_size=(11,11), stride=4)
        self.conv2 = nn.Conv2d(in_channels=96, out_channels=256, kernel_size=(5,5), padding=2)
        self.conv3 = nn.Conv2d(in_channels=256, out_channels=384, kernel_size=(3,3), padding=1)
        self.conv4 = nn.Conv2d(in_channels=384, out_channels=384, kernel_size=(3,3), padding=1)
        self.conv5 = nn.Conv2d(in_channels=384, out_channels=256, kernel_size=(3,3), padding=1)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(in_features=50176, out_features=4096)
        self.fc2 = nn.Linear(in_features=4096, out_features=4096)
        self.fc3 = nn.Linear(4096, 1)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.max_pool2d(x, kernel_size=(3,3), stride=2)
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, kernel_size=(3,3), stride=2)
        x = F.relu(self.conv3(x))
        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = F.max_pool2d(x, kernel_size=(3,3), stride=2)
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [16]:
model = AlexNet()
model.to(device)

AlexNet(
  (conv1): Conv2d(1, 96, kernel_size=(11, 11), stride=(4, 4))
  (conv2): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (conv3): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv5): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=50176, out_features=4096, bias=True)
  (fc2): Linear(in_features=4096, out_features=4096, bias=True)
  (fc3): Linear(in_features=4096, out_features=1, bias=True)
)

In [17]:
dummy = torch.tensor(np.zeros((16,1,500,500))).float().to(device)

summary(model=model, input_data=dummy)

Layer (type:depth-idx)                   Output Shape              Param #
AlexNet                                  [16, 1]                   --
├─Conv2d: 1-1                            [16, 96, 123, 123]        11,712
├─Conv2d: 1-2                            [16, 256, 61, 61]         614,656
├─Conv2d: 1-3                            [16, 384, 30, 30]         885,120
├─Conv2d: 1-4                            [16, 384, 30, 30]         1,327,488
├─Conv2d: 1-5                            [16, 256, 30, 30]         884,992
├─Flatten: 1-6                           [16, 50176]               --
├─Linear: 1-7                            [16, 4096]                205,524,992
├─Linear: 1-8                            [16, 4096]                16,781,312
├─Linear: 1-9                            [16, 1]                   4,097
Total params: 226,034,369
Trainable params: 226,034,369
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 87.59
Input size (MB): 16.00
Forward/backward pass size (MB): 42

In [18]:
data = pd.read_csv("../data/chest_xray/chest_xray_dataset.csv")

In [19]:
train_data = data[data['split'] == 'train']

In [20]:
train_data.iloc[0]['path']

'data/chest_xray/train/NORMAL/IM-0115-0001.jpeg'

In [21]:
from PIL import Image
import os 

class XrayDataset(Dataset):
    def __init__(self, split, transform):
        self.data = data[data['split'] == split]

        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        data_item = self.data.iloc[idx]
        label = self.data.iloc[idx]['class']
        image_data = Image.open(os.path.join("..", data_item['path']))
        image_data = self.transform(image_data)
        return image_data, label

In [22]:
train_transform = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=1),
    v2.ToDtype(torch.uint8, scale=True),
    v2.RandomResizedCrop(size=(500, 500)),
    v2.RandomHorizontalFlip(),
    v2.ToDtype(torch.float32, scale=True),
])

test_transform = v2.Compose([
    v2.ToImage(),
    v2.Grayscale(num_output_channels=1),
    v2.ToDtype(torch.uint8, scale=True),
    v2.CenterCrop(size=(500, 500)),
    v2.ToDtype(torch.float32, scale=True),
])

In [ ]:
train_dataset = XrayDataset(split='train', transform=train_transform)
val_dataset = XrayDataset(split="val", transform=test_transform)
test_dataset = XrayDataset(split="test", transform=test_transform)

In [ ]:
train_dataloader = DataLoader(dataset=train_dataset, shuffle=True, batch_size=64)
val_dataloader = DataLoader(dataset=val_dataset, shuffle=False, batch_size=64)
test_dataloader = DataLoader(dataset=test_dataset, shuffle=False, batch_size=64)

In [25]:
len(train_dataloader)

82

In [ ]:
epochs = 15
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), 0.001)

for epoch in range(epochs):
    epoch_loss = 0
    print(f"Epoch {epoch}:")
    model.train()
    for idx, (x, target) in enumerate(train_dataloader):
        input = x.to(device)
        target = target.to(device)
        output = model(input)
        output = output.squeeze()
        loss = criterion(output, target.float())
        # zero the gradients
        optimizer.zero_grad()
        # calculate the gradients based on the calculated loss (the gradient of the loss wrt each param)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    # Validation loss
    print("Loss", epoch_loss/len(train_dataloader))
    
    model.eval()
    with torch.no_grad():
        val_loss = 0
        all_preds = []
        all_targets = []

        for x, target in val_dataloader:
            x = x.to(device)
            target = target.to(device).float()

            output = model(x).squeeze()

            loss = criterion(output, target)
            val_loss += loss.item()

            preds = torch.sigmoid(output)
            all_preds.extend(preds.cpu())
            all_targets.extend(target.cpu())

        all_preds = torch.stack(all_preds)
        all_targets = torch.stack(all_targets)

        output_thresholded = (all_preds >= 0.5).numpy()
        accuracy = accuracy_score(all_targets.numpy(), output_thresholded)

        print(f"Val loss: {val_loss / len(val_dataloader)} | Val accuracy: {accuracy}")

Epoch 0:
431680.3457462809
Val loss: 6.268062114715576 | Val accuracy: 0.5
Epoch 1:
1.655951639137617
Val loss: 0.7957960963249207 | Val accuracy: 0.5
Epoch 2:
0.598982671412026
Val loss: 0.9123601317405701 | Val accuracy: 0.5
Epoch 3:
0.5799196489700457
Val loss: 0.8441650867462158 | Val accuracy: 0.5
Epoch 4:
0.5946588468987767
Val loss: 0.7526729702949524 | Val accuracy: 0.5625
Epoch 5:


In [ ]:
torch.tensor(3).long()

tensor(3)

In [ ]:
model.eval()
with torch.no_grad():
    test_loss = 0
    all_preds = []
    all_targets = []

    for x, target in test_dataloader:
        x = x.to(device)
        target = target.to(device).float()

        output = model(x).squeeze()

        loss = criterion(output, target)
        test_loss += loss.item()

        preds = torch.sigmoid(output)
        all_preds.extend(preds.cpu())
        all_targets.extend(target.cpu())

    all_preds = torch.stack(all_preds)
    all_targets = torch.stack(all_targets)

    output_thresholded = (all_preds >= 0.5).numpy()
    accuracy = accuracy_score(all_targets.numpy(), output_thresholded)

    print(f"Test loss: {test_loss / len(test_dataloader)} | Test accuracy: {accuracy}")

Test loss: 84198.71875 | Test accuracy: 0.9166666666666666
